# Effective barrier height $\Phi$ and exchange splitting $\Delta_{\mathrm{ex}}$ at 100 K

T-sweep counterpart to the 20 K analysis (`Barrier_vs_canting_angle_20K_v2.ipynb`). Same Gaussian + linear background fit on the Feenstra-normalised differential conductance $(dI/dV)/(I/V)$, but with `sigma_max` widened to 0.50 V (was 0.30 V at 20 K) so the broader peaks above 30 K do not get bound-rejected. `H_sat` (c-axis) and `H_sf` (b-axis) are estimated *per T* from the I(|H|) hinge / tanh fit in `scripts.saturation_field` rather than hard-coded, since both fields shift with T.

$T = 100$ K is below $T_{N}$; the AFM/FM constant fits and the canting model should both apply, though as $T \to T_{N}$ the AFM peak in $(dI/dV)/(I/V)$ broadens and the Gaussian fit acceptance drops.

**Outputs.** Phi(theta) per-bin table and Phi_AFM / Phi_FM / Delta_ex fit summary for each axis are written to `output/IV_H_scans/barrier_vs_canting_v2/{c,b}_scans/`. Where the fit cannot recover a clean peak the relevant entries are NaN, so the downstream summary notebook can skip those points.


## 1. Setup and data loading

In [ ]:
from scripts.utils import setup_notebook, OKABE_ITO_CYCLE
PROJECT_ROOT, np, pd, plt, Path = setup_notebook()

from scripts.IV_Hscan_gaussian import load_dataframe
from scripts.barrier_canting_fit import (
    normalised_didv, gauss_lin, fit_band_edge_gauss, extract_peaks,
    weighted_mean, bin_weighted, canting_angle, sin2_half,
    SIGMA_BOUNDS, HALF_WIDTH,
)
from scripts.saturation_field import (
    estimate_H_sat_c, estimate_H_sf_b, cross_check_phi_plateau,
)

TEMPERATURE = 100
df_c_path = PROJECT_ROOT / "output" / "IV_H_scans" / "dataframes" / "c_scans" / f"IV_gaussian_{TEMPERATURE}K.pkl"
df_b_path = PROJECT_ROOT / "output" / "IV_H_scans" / "dataframes" / "b_scans" / f"IV_gaussian_{TEMPERATURE}K.pkl"
df_c = load_dataframe(df_c_path)
df_b = load_dataframe(df_b_path)

print(f"c-axis: {len(df_c)} rows, H range {df_c['H'].min():+.3f} to {df_c['H'].max():+.3f} T")
print(f"b-axis: {len(df_b)} rows, H range {df_b['H'].min():+.3f} to {df_b['H'].max():+.3f} T")
print(f"Fit params: HALF_WIDTH={HALF_WIDTH}, SIGMA_BOUNDS={SIGMA_BOUNDS}")


## 2. Saturation and spin-flip fields from I(|H|)

Estimate `H_sat` (c-axis) and `H_sf` (b-axis) by fitting the symmetrised I(|H|) at a constant bias voltage (`V_bias = 0.6` V). The c-axis uses a hinge model (linear rise then plateau); the b-axis uses a tanh sigmoid centred at `H_sf`. Both fits are inspected visually below. The c-axis estimate is then cross-checked against the field where the binned Phi(|H_z|) plateau begins.


In [ ]:
V_BIAS = 0.6  # V; chosen in the FM-saturated rise of I(|H|)

sat_c = estimate_H_sat_c(df_c, V_bias=V_BIAS, H_sat_init=2.0)
sat_b = estimate_H_sf_b(df_b,  V_bias=V_BIAS, H_sf_init=0.3)

H_SAT_C = sat_c['H_sat'] if sat_c['ok'] else 2.0
H_SF_B  = sat_b['H_sf']  if sat_b['ok'] else 0.3

print(f"c-axis H_sat = ({sat_c['H_sat']:.3f} +/- {sat_c['H_sat_err']:.3f}) T   ok={sat_c['ok']}")
print(f"b-axis H_sf  = ({sat_b['H_sf']:.3f}  +/- {sat_b['H_sf_err']:.3f}) T   ok={sat_b['ok']}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, out, label, key in [(axes[0], sat_c, fr'$c$-axis: $H_{{\mathrm{{sat}}}}={sat_c["H_sat"]:.2f}$ T', 'H_sat'),
                            (axes[1], sat_b, fr'$b$-axis: $H_{{\mathrm{{sf}}}}={sat_b["H_sf"]:.2f}$ T',  'H_sf')]:
    ax.plot(out['h'], 1e6 * np.asarray(out['I_avg']), 'o', color=OKABE_ITO_CYCLE[1], markersize=4, label='data (avg)')
    if out['fit_h'] is not None:
        ax.plot(out['fit_h'], 1e6 * np.asarray(out['fit_I']), '-', color=OKABE_ITO_CYCLE[5], lw=1.6, label='fit')
        ax.axvline(out[key], color='k', ls='--', lw=0.8)
    ax.set_xlabel(r'$|H|$ (T)')
    ax.set_ylabel(r'$I$ ($\mu$A)')
    ax.set_title(label)
    ax.legend(loc='best', fontsize=12)
fig.tight_layout()
plt.show()


## 3. Spot-check the Gaussian + linear fits

Plot the normalised dI/dV with overlaid fits at a handful of representative fields, to verify the peak position and width evolve sensibly with H before running the full extraction.


In [ ]:
def plot_spot_check(df, target_fields, axis_label):
    fig, ax = plt.subplots(figsize=(6, 5), dpi=200)
    for i, H_target in enumerate(target_fields):
        idx = (df['H'] - H_target).abs().idxmin()
        row = df.loc[idx]
        V, norm = normalised_didv(row['voltage_smooth'], row['current_smooth'])
        g = fit_band_edge_gauss(V, norm)
        color = OKABE_ITO_CYCLE[i % len(OKABE_ITO_CYCLE)]
        ax.plot(V, norm, 'o', color=color, markersize=2.5, alpha=0.45,
                label=fr'${axis_label}={row["H"]:+.2f}$ T')
        if g['popt'] is not None:
            Vmin, Vmax = g['window']
            Vfit = np.linspace(Vmin, Vmax, 400)
            ax.plot(Vfit, gauss_lin(Vfit, *g['popt']), '-', color=color, lw=1.4)
        if g['ok'] and np.isfinite(g['V_peak']):
            ax.axvline(g['V_peak'], color=color, ls='--', lw=0.9, alpha=0.85)
    ax.set_xlabel(r'$V_{\mathrm{bias}}$ (V)')
    ax.set_ylabel(r'$(dI/dV)/(I/V)$')
    ax.set_xlim(-0.05, 1.0)
    ax.legend(loc='upper left', fontsize=8, frameon=True)
    fig.tight_layout()
    plt.show()


plot_spot_check(df_c, [0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0], axis_label='H_Z')
plot_spot_check(df_b, [0.0, 0.10, 0.15, 0.50, 0.70, 0.90],                axis_label='H_y')


## 4. Extract $V_{\mathrm{peak}}$ for every $I(V)$ curve


In [ ]:
res_c = extract_peaks(df_c)
res_b = extract_peaks(df_b)
res_c['theta_deg'] = np.degrees(canting_angle(res_c['H'].values, H_SAT_C))
res_c['sin2_half'] = sin2_half(res_c['H'].values, H_SAT_C)

print(f"c-axis: {len(res_c)} / {len(df_c)} curves accepted "
      f"(median uncertainty {1000 * res_c['Phi_err_eV'].median():.2f} meV)" if len(res_c) else
      "c-axis: 0 / {len(df_c)} curves accepted")
print(f"b-axis: {len(res_b)} / {len(df_b)} curves accepted "
      f"(median uncertainty {1000 * res_b['Phi_err_eV'].median():.2f} meV)" if len(res_b) else
      f"b-axis: 0 / {len(df_b)} curves accepted")


## 5. $c$-axis $\Phi(H_z)$ with endpoint constant fits

`H_AFM_C_MAX` is held at 0.10 T (pure AFM, no observable canting); `H_FM_C_MIN` is set just inside the fitted `H_sat` so the FM endpoint sees only saturated points.


In [ ]:
H_AFM_C_MAX = 0.10
H_FM_C_MIN  = max(H_SAT_C - 0.10, 0.5 * H_SAT_C)

if len(res_c):
    afm_c_mask = res_c['abs_H'] < H_AFM_C_MAX
    fm_c_mask  = res_c['abs_H'] > H_FM_C_MIN

    Phi_AFM_c, Phi_AFM_c_err, n_AFM_c = weighted_mean(res_c.loc[afm_c_mask, 'Phi_eV'],
                                                      res_c.loc[afm_c_mask, 'Phi_err_eV'])
    Phi_FM_c,  Phi_FM_c_err,  n_FM_c  = weighted_mean(res_c.loc[fm_c_mask,  'Phi_eV'],
                                                      res_c.loc[fm_c_mask,  'Phi_err_eV'])
    Delta_c     = Phi_AFM_c - Phi_FM_c
    Delta_c_err = float(np.sqrt(Phi_AFM_c_err**2 + Phi_FM_c_err**2)) if np.isfinite(Phi_AFM_c) and np.isfinite(Phi_FM_c) else np.nan
    agg_c_H = bin_weighted(res_c, 'H', round_decimals=2)
else:
    Phi_AFM_c = Phi_AFM_c_err = Phi_FM_c = Phi_FM_c_err = Delta_c = Delta_c_err = np.nan
    n_AFM_c = n_FM_c = 0
    afm_c_mask = fm_c_mask = pd.Series([], dtype=bool)
    agg_c_H = pd.DataFrame()

fig, ax = plt.subplots(figsize=(6, 5), dpi=200)
if len(res_c):
    excl_c = ~(afm_c_mask | fm_c_mask)
    ax.errorbar(res_c.loc[afm_c_mask, 'H'], 1000 * res_c.loc[afm_c_mask, 'Phi_eV'],
                yerr=1000 * res_c.loc[afm_c_mask, 'Phi_err_eV'],
                fmt='o', color=OKABE_ITO_CYCLE[1], markersize=4, alpha=0.85,
                ecolor=OKABE_ITO_CYCLE[1], elinewidth=0.7, capsize=2,
                label=fr'AFM ($|H_z|<{H_AFM_C_MAX:.2f}$ T), $n={n_AFM_c}$')
    ax.errorbar(res_c.loc[fm_c_mask, 'H'], 1000 * res_c.loc[fm_c_mask, 'Phi_eV'],
                yerr=1000 * res_c.loc[fm_c_mask, 'Phi_err_eV'],
                fmt='s', color=OKABE_ITO_CYCLE[5], markersize=4, alpha=0.85,
                ecolor=OKABE_ITO_CYCLE[5], elinewidth=0.7, capsize=2,
                label=fr'FM ($|H_z|>{H_FM_C_MIN:.2f}$ T), $n={n_FM_c}$')
    ax.errorbar(res_c.loc[excl_c, 'H'], 1000 * res_c.loc[excl_c, 'Phi_eV'],
                yerr=1000 * res_c.loc[excl_c, 'Phi_err_eV'],
                fmt='x', color='0.55', markersize=4, alpha=0.55,
                ecolor='0.55', elinewidth=0.5, capsize=1, label='canting (excluded)')

    for mu, sig, mask, color in [(Phi_AFM_c, Phi_AFM_c_err, afm_c_mask, OKABE_ITO_CYCLE[1]),
                                 (Phi_FM_c,  Phi_FM_c_err,  fm_c_mask,  OKABE_ITO_CYCLE[5])]:
        if not np.isfinite(mu):
            continue
        for branch in [res_c.loc[mask & (res_c['H'] < 0), 'H'],
                       res_c.loc[mask & (res_c['H'] > 0), 'H']]:
            if branch.empty:
                continue
            ax.hlines(1000 * mu, branch.min(), branch.max(), color=color, lw=1.6)
            ax.fill_between([branch.min(), branch.max()],
                            1000 * (mu - sig), 1000 * (mu + sig),
                            color=color, alpha=0.25, lw=0)
ax.set_xlabel(r'$H_Z$ (T)')
ax.set_ylabel(r'$\Phi$ (meV)')
ax.legend(loc='lower right', fontsize=8)
if np.isfinite(Phi_AFM_c) and np.isfinite(Phi_FM_c):
    ax.text(0.03, 0.97,
            fr'$\Phi_{{\mathrm{{AFM}}}}=({1000 * Phi_AFM_c:.1f}\pm{1000 * Phi_AFM_c_err:.1f})$ meV''\n'
            fr'$\Phi_{{\mathrm{{FM}}}}\,=\,({1000 * Phi_FM_c:.1f}\pm{1000 * Phi_FM_c_err:.1f})$ meV''\n'
            fr'$\Delta_{{\mathrm{{ex}}}}\,=\,({1000 * Delta_c:.1f}\pm{1000 * Delta_c_err:.1f})$ meV',
            transform=ax.transAxes, va='top', ha='left', fontsize=10,
            bbox=dict(facecolor='white', edgecolor='0.7', alpha=0.9))
fig.tight_layout()
plt.show()

print(f"c-axis Phi_AFM  = ({1000 * Phi_AFM_c:.2f} +/- {1000 * Phi_AFM_c_err:.2f}) meV  (n={n_AFM_c})")
print(f"c-axis Phi_FM   = ({1000 * Phi_FM_c:.2f} +/- {1000 * Phi_FM_c_err:.2f}) meV  (n={n_FM_c})")
print(f"c-axis Delta_ex = ({1000 * Delta_c:.2f} +/- {1000 * Delta_c_err:.2f}) meV")


## 6. $\Phi$ vs canting angle $\theta$


In [ ]:
if len(res_c):
    res_c_pos = res_c.copy()
    res_c_pos['abs_H_eff'] = np.minimum(res_c_pos['abs_H'].values, H_SAT_C)
    agg_c_theta = bin_weighted(res_c_pos.assign(abs_H=res_c_pos['abs_H_eff']),
                               'abs_H', round_decimals=2)
    agg_c_theta['theta_deg'] = np.degrees(canting_angle(agg_c_theta['abs_H'].values, H_SAT_C))
    agg_c_theta['sin2_half'] = sin2_half(agg_c_theta['abs_H'].values, H_SAT_C)

    fig, ax = plt.subplots(figsize=(6, 5), dpi=200)
    ax.errorbar(agg_c_theta['theta_deg'], 1000 * agg_c_theta['Phi_eV'],
                yerr=1000 * agg_c_theta['Phi_err_eV'],
                fmt='o', color=OKABE_ITO_CYCLE[2], markersize=4,
                ecolor=OKABE_ITO_CYCLE[2], elinewidth=0.8, capsize=2, alpha=0.9)
    ax.set_xlabel(r'Interlayer angle $\theta$ (deg)')
    ax.set_ylabel(r'$\Phi(\theta)$ (meV)')
    ax.set_xticks([0, 45, 90, 135, 180])
    fig.tight_layout()
    plt.show()
    print(f'{len(agg_c_theta)} theta bins')
else:
    agg_c_theta = pd.DataFrame()
    print('No accepted c-axis curves at this T; skipping Phi(theta) panel.')


## 7. Alternating-barrier test: $\Phi$ vs $\sin^2(\theta/2)$

Inverse-variance weighted linear regression, with Birge rescale if $\chi^2_{\mathrm{red}}>1$. Reports `chi2_red`, `Phi_FM_lin`, `Phi_AFM_lin`, `Delta_lin` so the summary notebook can plot the alternating-barrier slope as a function of T.


In [ ]:
if len(agg_c_theta) >= 3:
    x    = agg_c_theta['sin2_half'].values
    y    = agg_c_theta['Phi_eV'].values
    yerr = agg_c_theta['Phi_err_eV'].values
    pos  = yerr > 0
    safe = np.where(pos, yerr, np.median(yerr[pos]) if pos.any() else 1.0)
    w    = 1.0 / safe**2

    S, Sx   = w.sum(), (w * x).sum()
    Sxx     = (w * x * x).sum()
    Sy, Sxy = (w * y).sum(), (w * x * y).sum()
    det     = S * Sxx - Sx * Sx
    a_lin   = (Sxx * Sy - Sx * Sxy) / det
    b_lin   = (S * Sxy - Sx * Sy) / det
    var_a, var_b, cov_ab = Sxx / det, S / det, -Sx / det

    resid = y - (a_lin + b_lin * x)
    dof = max(x.size - 2, 1)
    chi2_red_lin = float(np.sum(w * resid**2) / dof)
    scale = max(1.0, chi2_red_lin)
    var_a  *= scale
    var_b  *= scale
    cov_ab *= scale

    Phi_FM_c_lin      = a_lin
    Phi_FM_c_lin_err  = float(np.sqrt(var_a))
    Delta_c_lin       = b_lin
    Delta_c_lin_err   = float(np.sqrt(var_b))
    Phi_AFM_c_lin     = a_lin + b_lin
    Phi_AFM_c_lin_err = float(np.sqrt(var_a + var_b + 2 * cov_ab))

    xx = np.linspace(0.0, 1.0, 200)
    yy = a_lin + b_lin * xx
    band = np.sqrt(var_a + 2 * xx * cov_ab + xx**2 * var_b)

    fig, ax = plt.subplots(figsize=(6, 5), dpi=200)
    ax.errorbar(x, 1000 * y, yerr=1000 * yerr, fmt='o',
                color=OKABE_ITO_CYCLE[2], markersize=4, alpha=0.9,
                ecolor=OKABE_ITO_CYCLE[2], elinewidth=0.8, capsize=2, label='c-axis data')
    ax.plot(xx, 1000 * yy, '-', color='black', lw=1.3, label='Linear fit')
    ax.fill_between(xx, 1000 * (yy - band), 1000 * (yy + band), color='black', alpha=0.12, lw=0)
    ax.set_xlabel(r'$\sin^2(\theta/2) = 1 - (H_Z/H_{\mathrm{sat}})^2$')
    ax.set_ylabel(r'$\Phi(\theta)$ (meV)')
    ax.set_xlim(-0.05, 1.05)
    ax.legend(loc='lower right', fontsize=12)
    fig.tight_layout()
    plt.show()

    print(f'chi^2_red   = {chi2_red_lin:.2f}  (Birge scale: {scale:.2f})')
    print(f'Phi_FM_lin  = ({1000 * Phi_FM_c_lin:.2f} +/- {1000 * Phi_FM_c_lin_err:.2f}) meV')
    print(f'Phi_AFM_lin = ({1000 * Phi_AFM_c_lin:.2f} +/- {1000 * Phi_AFM_c_lin_err:.2f}) meV')
    print(f'Delta_lin   = ({1000 * Delta_c_lin:.2f} +/- {1000 * Delta_c_lin_err:.2f}) meV')
else:
    Phi_AFM_c_lin = Phi_AFM_c_lin_err = Phi_FM_c_lin = Phi_FM_c_lin_err = np.nan
    Delta_c_lin = Delta_c_lin_err = chi2_red_lin = np.nan
    print('Too few theta bins for a linear fit.')


## 8. $b$-axis two-state analysis

The b-axis spin-flip is sharp around `H_sf` (fitted in section 2). AFM endpoint at $|H_y| < 0.7\,H_{\mathrm{sf}}$, FM endpoint at $|H_y| > 1.3\,H_{\mathrm{sf}}$ (hard-clamped to the data range). The transition window in between is excluded.


In [ ]:
H_AFM_B_MAX = 0.7 * H_SF_B
H_FM_B_MIN  = min(1.3 * H_SF_B, df_b['H'].abs().max() - 0.05)

if len(res_b):
    afm_b_mask = res_b['abs_H'] < H_AFM_B_MAX
    fm_b_mask  = res_b['abs_H'] > H_FM_B_MIN

    Phi_AFM_b, Phi_AFM_b_err, n_AFM_b = weighted_mean(res_b.loc[afm_b_mask, 'Phi_eV'],
                                                      res_b.loc[afm_b_mask, 'Phi_err_eV'])
    Phi_FM_b,  Phi_FM_b_err,  n_FM_b  = weighted_mean(res_b.loc[fm_b_mask,  'Phi_eV'],
                                                      res_b.loc[fm_b_mask,  'Phi_err_eV'])
    Delta_b     = Phi_AFM_b - Phi_FM_b
    Delta_b_err = float(np.sqrt(Phi_AFM_b_err**2 + Phi_FM_b_err**2)) if np.isfinite(Phi_AFM_b) and np.isfinite(Phi_FM_b) else np.nan
else:
    Phi_AFM_b = Phi_AFM_b_err = Phi_FM_b = Phi_FM_b_err = Delta_b = Delta_b_err = np.nan
    n_AFM_b = n_FM_b = 0
    afm_b_mask = fm_b_mask = pd.Series([], dtype=bool)

fig, ax = plt.subplots(figsize=(6, 5), dpi=200)
if len(res_b):
    excl_b = ~(afm_b_mask | fm_b_mask)
    ax.errorbar(res_b.loc[afm_b_mask, 'H'], 1000 * res_b.loc[afm_b_mask, 'Phi_eV'],
                yerr=1000 * res_b.loc[afm_b_mask, 'Phi_err_eV'],
                fmt='o', color=OKABE_ITO_CYCLE[1], markersize=4, alpha=0.85,
                ecolor=OKABE_ITO_CYCLE[1], elinewidth=0.7, capsize=2,
                label=fr'AFM ($|H_y|<{H_AFM_B_MAX:.2f}$ T), $n={n_AFM_b}$')
    ax.errorbar(res_b.loc[fm_b_mask, 'H'], 1000 * res_b.loc[fm_b_mask, 'Phi_eV'],
                yerr=1000 * res_b.loc[fm_b_mask, 'Phi_err_eV'],
                fmt='s', color=OKABE_ITO_CYCLE[5], markersize=4, alpha=0.85,
                ecolor=OKABE_ITO_CYCLE[5], elinewidth=0.7, capsize=2,
                label=fr'FM ($|H_y|>{H_FM_B_MIN:.2f}$ T), $n={n_FM_b}$')
    ax.errorbar(res_b.loc[excl_b, 'H'], 1000 * res_b.loc[excl_b, 'Phi_eV'],
                yerr=1000 * res_b.loc[excl_b, 'Phi_err_eV'],
                fmt='x', color='0.55', markersize=4, alpha=0.55,
                ecolor='0.55', elinewidth=0.5, capsize=1, label='spin-flip (excluded)')

    for sign in (-1, +1):
        ax.axvspan(sign * H_AFM_B_MAX, sign * H_FM_B_MIN, color='0.85', alpha=0.45, lw=0)

    for mask, mu, sig, color in [(afm_b_mask, Phi_AFM_b, Phi_AFM_b_err, OKABE_ITO_CYCLE[1]),
                                  (fm_b_mask, Phi_FM_b, Phi_FM_b_err, OKABE_ITO_CYCLE[5])]:
        if not np.isfinite(mu):
            continue
        Hs = res_b.loc[mask, 'H']
        for branch in [Hs[Hs < 0], Hs[Hs > 0]]:
            if branch.empty:
                continue
            ax.hlines(1000 * mu, branch.min(), branch.max(), color=color, lw=1.6)
            ax.fill_between([branch.min(), branch.max()], 1000 * (mu - sig), 1000 * (mu + sig),
                            color=color, alpha=0.25, lw=0)

ax.set_xlabel(r'$H_y$ (T)')
ax.set_ylabel(r'$\Phi$ (meV)')
ax.legend(loc='lower right', fontsize=8)
if np.isfinite(Phi_AFM_b) and np.isfinite(Phi_FM_b):
    ax.text(0.03, 0.97,
            fr'$\Phi_{{\mathrm{{AFM}}}}=({1000 * Phi_AFM_b:.1f}\pm{1000 * Phi_AFM_b_err:.1f})$ meV''\n'
            fr'$\Phi_{{\mathrm{{FM}}}}\,=\,({1000 * Phi_FM_b:.1f}\pm{1000 * Phi_FM_b_err:.1f})$ meV''\n'
            fr'$\Delta_{{\mathrm{{ex}}}}\,=\,({1000 * Delta_b:.1f}\pm{1000 * Delta_b_err:.1f})$ meV',
            transform=ax.transAxes, va='top', ha='left', fontsize=10,
            bbox=dict(facecolor='white', edgecolor='0.7', alpha=0.9))
fig.tight_layout()
plt.show()

print(f"b-axis Phi_AFM  = ({1000 * Phi_AFM_b:.2f} +/- {1000 * Phi_AFM_b_err:.2f}) meV  (n={n_AFM_b})")
print(f"b-axis Phi_FM   = ({1000 * Phi_FM_b:.2f} +/- {1000 * Phi_FM_b_err:.2f}) meV  (n={n_FM_b})")
print(f"b-axis Delta_ex = ({1000 * Delta_b:.2f} +/- {1000 * Delta_b_err:.2f}) meV")


## 9. Save deliverables

Outputs go under `output/IV_H_scans/barrier_vs_canting_v2/{c,b}_scans/`, matching the 20 K v2 file naming so the summary notebook can globalise across T with a single pattern.


In [ ]:
out_c = PROJECT_ROOT / 'output' / 'IV_H_scans' / 'barrier_vs_canting_v2' / 'c_scans'
out_b = PROJECT_ROOT / 'output' / 'IV_H_scans' / 'barrier_vs_canting_v2' / 'b_scans'
out_c.mkdir(parents=True, exist_ok=True)
out_b.mkdir(parents=True, exist_ok=True)

if len(agg_c_theta):
    agg_c_out = agg_c_theta.copy()
    agg_c_out['Phi_meV']     = 1000 * agg_c_out['Phi_eV']
    agg_c_out['Phi_meV_err'] = 1000 * agg_c_out['Phi_err_eV']
    agg_c_out.to_csv(out_c / f'Phi_vs_theta_{TEMPERATURE}K.csv', index=False)

pd.DataFrame([{
    'temperature_K':         TEMPERATURE,
    'H_sat_T':               H_SAT_C,
    'H_sat_T_err':           sat_c.get('H_sat_err', np.nan),
    'H_sat_ok':              sat_c.get('ok', False),
    'H_AFM_max_T':           H_AFM_C_MAX,
    'H_FM_min_T':            H_FM_C_MIN,
    'n_AFM':                 n_AFM_c,
    'n_FM':                  n_FM_c,
    'Phi_AFM_meV':           1000 * Phi_AFM_c,
    'Phi_AFM_err_meV':       1000 * Phi_AFM_c_err,
    'Phi_FM_meV':            1000 * Phi_FM_c,
    'Phi_FM_err_meV':        1000 * Phi_FM_c_err,
    'Delta_ex_meV':          1000 * Delta_c,
    'Delta_ex_err_meV':      1000 * Delta_c_err,
    'Phi_AFM_lin_meV':       1000 * Phi_AFM_c_lin,
    'Phi_AFM_lin_err_meV':   1000 * Phi_AFM_c_lin_err,
    'Phi_FM_lin_meV':        1000 * Phi_FM_c_lin,
    'Phi_FM_lin_err_meV':    1000 * Phi_FM_c_lin_err,
    'Delta_ex_lin_meV':      1000 * Delta_c_lin,
    'Delta_ex_lin_err_meV':  1000 * Delta_c_lin_err,
    'chi2_red_lin':          chi2_red_lin,
}]).to_csv(out_c / f'Phi_vs_theta_fit_{TEMPERATURE}K.csv', index=False)

if len(res_b):
    res_b_out = res_b.copy()
    res_b_out['Phi_meV']     = 1000 * res_b_out['Phi_eV']
    res_b_out['Phi_meV_err'] = 1000 * res_b_out['Phi_err_eV']
    res_b_out['window'] = np.where(afm_b_mask, 'AFM',
                           np.where(fm_b_mask, 'FM', 'excluded'))
    res_b_out.to_csv(out_b / f'Phi_vs_Hy_{TEMPERATURE}K.csv', index=False)

pd.DataFrame([{
    'temperature_K':    TEMPERATURE,
    'H_sf_T':           H_SF_B,
    'H_sf_T_err':       sat_b.get('H_sf_err', np.nan),
    'H_sf_ok':          sat_b.get('ok', False),
    'H_AFM_max_T':      H_AFM_B_MAX,
    'H_FM_min_T':       H_FM_B_MIN,
    'n_AFM':            n_AFM_b,
    'n_FM':             n_FM_b,
    'Phi_AFM_meV':      1000 * Phi_AFM_b,
    'Phi_AFM_err_meV':  1000 * Phi_AFM_b_err,
    'Phi_FM_meV':       1000 * Phi_FM_b,
    'Phi_FM_err_meV':   1000 * Phi_FM_b_err,
    'Delta_ex_meV':     1000 * Delta_b,
    'Delta_ex_err_meV': 1000 * Delta_b_err,
}]).to_csv(out_b / f'Phi_two_state_fit_{TEMPERATURE}K.csv', index=False)

print('Saved per-T outputs to', out_c, 'and', out_b)
